In [1]:
import pandas as pd
import numpy as np

In [2]:
df_deliveries = pd.read_csv('CSV/deliveries.csv')
df_matches = pd.read_csv('CSV/matches.csv')

In [53]:
print(df_deliveries.head(1))

   match_id  inning           batting_team                 bowling_team  over  \
0    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   

   ball      batter   bowler  non_striker  batsman_runs  extra_runs  \
0     1  SC Ganguly  P Kumar  BB McCullum             0           1   

   total_runs extras_type  is_wicket player_dismissed dismissal_kind fielder  
0           1     legbyes          0              NaN            NaN     NaN  


In [3]:
teams = [
    'Sunrisers Hyderabad', 'Mumbai Indians', 'Royal Challengers Bangaluru',
    'Kolkata Knight Riders', 'Punjab Kings', 'Chennai Super Kings',
    'Rajasthan Royals', 'Delhi Capitals', 'Gujarat Titans', 'Lucknow Super Giants'
]

In [4]:
df_matches['team1'] = df_matches['team1'].str.replace('Delhi Daredevils','Delhi Capitals')
df_matches['team2'] = df_matches['team2'].str.replace('Delhi Daredevils','Delhi Capitals')

df_matches['team1'] = df_matches['team1'].str.replace('Deccan Chargers','Sunrisers Hyderabad')
df_matches['team2'] = df_matches['team2'].str.replace('Deccan Chargers','Sunrisers Hyderabad')

df_matches['team1'] = df_matches['team1'].str.replace('Kings XI Punjab','Punjab Kings')
df_matches['team2'] = df_matches['team2'].str.replace('Kings XI Punjab','Punjab Kings')

df_matches['team1'] = df_matches['team1'].str.replace('Royal Challengers Bangalore','Royal Challengers Bengaluru')
df_matches['team2'] = df_matches['team2'].str.replace('Royal Challengers Bangalore','Royal Challengers Bengaluru')

In [5]:
df_matches = df_matches[df_matches['team1'].isin(teams)]
df_matches = df_matches[df_matches['team2'].isin(teams)]

In [6]:
df_matches = df_matches[df_matches['method'].isna()]

In [7]:
df_matches.groupby('city')['venue'].unique()

city
Abu Dhabi         [Sheikh Zayed Stadium, Zayed Cricket Stadium, ...
Ahmedabad         [Sardar Patel Stadium, Motera, Narendra Modi S...
Bangalore                                   [M Chinnaswamy Stadium]
Bloemfontein                                      [OUTsurance Oval]
Cape Town                                                [Newlands]
Centurion                                         [SuperSport Park]
Chandigarh        [Punjab Cricket Association Stadium, Mohali, P...
Chennai           [MA Chidambaram Stadium, Chepauk, MA Chidambar...
Cuttack                                          [Barabati Stadium]
Delhi             [Feroz Shah Kotla, Arun Jaitley Stadium, Arun ...
Dharamsala        [Himachal Pradesh Cricket Association Stadium,...
Dubai                         [Dubai International Cricket Stadium]
Durban                                                  [Kingsmead]
East London                                          [Buffalo Park]
Guwahati                      [Barsapara Cr

In [10]:
df_matches['venue_clean'] = (
    df_matches['venue']
    .str.strip()
    .str.lower()
    .str.split(",").str[0]
)

In [12]:
venue_mapping = {
    # Delhi
    "feroz shah kotla": "arun jaitley stadium",
    "arun jaitley stadium": "arun jaitley stadium",

    # Ahmedabad
    "sardar patel stadium": "narendra modi stadium",
    "motera": "narendra modi stadium",
    "narendra modi stadium": "narendra modi stadium",

    # Mohali
    "punjab cricket association stadium": "punjab cricket association is bindra stadium",
    "punjab cricket association is bindra stadium": "punjab cricket association is bindra stadium",

    # Pune
    "subrata roy sahara stadium": "maharashtra cricket association stadium",
    "maharashtra cricket association stadium": "maharashtra cricket association stadium",

    # Abu Dhabi
    "sheikh zayed stadium": "zayed cricket stadium",
    "zayed cricket stadium": "zayed cricket stadium",

    # Chennai
    "ma chidambaram stadium": "ma chidambaram stadium",

    # Mumbai
    "wankhede stadium": "wankhede stadium",
    "brabourne stadium": "brabourne stadium",
    "dr dy patil sports academy": "dr dy patil sports academy",

    # Hyderabad
    "rajiv gandhi international stadium": "rajiv gandhi international stadium",

    # Visakhapatnam
    "dr. y.s. rajasekhara reddy aca-vdca cricket stadium": "dr. y.s. rajasekhara reddy aca-vdca cricket stadium",

    # Jaipur
    "sawai mansingh stadium": "sawai mansingh stadium",

    # Nagpur
    "vidarbha cricket association stadium": "vidarbha cricket association stadium",

    # Lucknow
    "bharat ratna shri atal bihari vajpayee ekana cricket stadium": "bharat ratna shri atal bihari vajpayee ekana cricket stadium",

    # Ranchi
    "jsca international stadium complex": "jsca international stadium complex",

    # Raipur
    "shaheed veer narayan singh international stadium": "shaheed veer narayan singh international stadium",

    # Indore
    "holkar cricket stadium": "holkar cricket stadium",

    # Bangalore
    "m chinnaswamy stadium": "m chinnaswamy stadium",

    # Kolkata
    "eden gardens": "eden gardens",

    # International (South Africa / UAE)
    "newlands": "newlands",
    "kingsmead": "kingsmead",
    "supersport park": "supersport park",
    "st george's park": "st george's park",
    "new wanderers stadium": "new wanderers stadium",
    "buffalo park": "buffalo park",
    "outsurance oval": "outsurance oval",
    "de beers diamond oval": "de beers diamond oval",
    "barsapara cricket stadium": "barsapara cricket stadium",
    "sharjah cricket stadium": "sharjah cricket stadium",
    "dubai international cricket stadium": "dubai international cricket stadium",
    "maharaja yadavindra singh international cricket stadium": "maharaja yadavindra singh international cricket stadium",
}


In [13]:
df_matches['venue_clean'] = (
    df_matches['venue_clean']
    .str.strip().str.lower()
    .replace(venue_mapping)
)


In [14]:
df_matches.drop('venue', axis=1,inplace=True)
df_matches.rename(columns={'venue_clean':'venue'},inplace=True)

In [17]:
df_matches = df_matches[['id','venue','winner','target_runs']]

In [18]:
df_matches.rename(columns={'id' : 'match_id'},inplace=True)

C:\Users\shres\AppData\Local\Temp\ipykernel_32412\3260332357.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_matches.rename(columns={'id' : 'match_id'},inplace=True)


In [19]:
delivery = df_matches.merge(df_deliveries, on='match_id')

In [20]:
delivery = delivery[delivery['inning'] == 2]

In [22]:
delivery['current_score'] = delivery.groupby('match_id')['total_runs'].cumsum()

In [23]:
delivery['runs_left'] =  delivery['target_runs'] - delivery['current_score']

In [24]:
delivery['balls_left'] =  120 - (delivery['over'] * 6 + delivery['ball'])

In [26]:
wickets = delivery.groupby('match_id')['is_wicket'].cumsum()
delivery['wickets_left'] = 10 - wickets

In [27]:
delivery['crr'] = delivery['current_score'] * 6 / (120- delivery['balls_left'])

In [28]:
delivery['rrr'] = delivery['runs_left'] * 6 / delivery['balls_left']

In [29]:
def result(row):
    return 1 if row['batting_team'] == row['winner'] else 0

In [30]:
delivery['result'] = delivery.apply(result,axis=1)

In [32]:
final_df = delivery[['match_id','batting_team','bowling_team','venue','runs_left','balls_left','wickets_left','target_runs','crr','rrr','result']]

In [33]:
final_df = final_df[final_df['balls_left'] != 0]

In [34]:
final_df.dropna(subset=['rrr'],inplace=True)

In [55]:
final_df.columns

Index(['match_id', 'batting_team', 'bowling_team', 'venue', 'runs_left',
       'balls_left', 'wickets_left', 'target_runs', 'crr', 'rrr', 'result'],
      dtype='object')

In [67]:
print(final_df.head(1))

     match_id     batting_team         bowling_team  \
124    335983  Kings XI Punjab  Chennai Super Kings   

                                            venue  runs_left  balls_left  \
124  punjab cricket association is bindra stadium      237.0         119   

     wickets_left  target_runs   crr       rrr  result current_phase  \
124            10        241.0  24.0  11.94958       0     powerplay   

     over_completed  
124        0.166667  


In [ ]:
def phase_and_progress(row, total_overs=20):
    balls_bowled = total_overs*6 - row['balls_left']
    
    if row['balls_left'] <= 30:
        phase = 'death_over'
    elif row['balls_left'] >= 84:
        phase = 'powerplay'
    else:
        phase = 'middle_over'
    
    overs_completed = balls_bowled / 6
    
    return pd.Series([phase, overs_completed], index=['phase', 'overs_completed'])

In [66]:
final_df[['current_phase','over_completed']] = final_df.apply(phase_and_progress, axis=1)

In [37]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [38]:
unique_matches = final_df['match_id'].unique()
train_matches, test_matches = train_test_split(
    unique_matches, test_size=0.2, random_state=2
)

In [41]:
train_data = final_df[final_df['match_id'].isin(train_matches)]
test_data = final_df[final_df['match_id'].isin(test_matches)]

In [42]:
train_data['match_id'].value_counts().sort_values()

match_id
336021      38
1082626     49
1254093     53
1426312     67
1304078     69
          ... 
1426294    130
1426288    131
1370350    131
1304076    131
1359480    135
Name: count, Length: 580, dtype: int64

In [43]:
test_data['match_id'].value_counts().sort_values()

match_id
1426270     55
1426295     60
548307      68
1082635     83
1178413     94
          ... 
1254069    128
419142     128
1304053    129
829777     130
829811     133
Name: count, Length: 146, dtype: int64

In [44]:
train_data = train_data.sample(frac = 1, random_state=42).reset_index(drop=True)
test_data = test_data.sample(frac = 1, random_state=42).reset_index(drop=True)

In [45]:
X_train = train_data.drop(columns=['result','match_id'])
y_train = train_data['result']

X_test = test_data.drop(columns=['result','match_id'])
y_test = test_data['result']

In [46]:
trf = ColumnTransformer(
    transformers=[
        ('trf', OneHotEncoder(sparse_output=False, drop='first'), ['batting_team', 'bowling_team', 'venue'])
    ], remainder='passthrough'
)

In [47]:
pipe = Pipeline(steps=[
    ('step1',trf),
    ('step2',LogisticRegression(solver='liblinear'))
])

In [50]:
pipe.fit(X_train,y_train)

,steps,"[('step1', ...), ('step2', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('trf', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [51]:
scores = cross_val_score(pipe,X_train,y_train,cv = 10, scoring = 'accuracy' )
print(f'CV Accuracy scores: {scores}')
print(f'mean : {scores.mean()}')

CV Accuracy scores: [0.81904339 0.82126462 0.81400859 0.82183057 0.82153436 0.81901659
 0.82508886 0.81768365 0.8264218  0.82834716]
mean : 0.8214239585811608


In [52]:
y_pred = pipe.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, y_pred))


print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

print("Classification Report:\n", classification_report(y_test, y_pred))


Test Accuracy: 0.7530588235294118
Confusion Matrix:
 [[6371 2215]
 [1983 6431]]
Classification Report:
               precision    recall  f1-score   support

           0       0.76      0.74      0.75      8586
           1       0.74      0.76      0.75      8414

    accuracy                           0.75     17000
   macro avg       0.75      0.75      0.75     17000
weighted avg       0.75      0.75      0.75     17000

